# Bolt YOLOv8 Segmentation Training

Colab에서 `pose_dataset.zip`을 변환하고 YOLOv8n-seg를 학습합니다. 먼저 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요.

In [ ]:
!pip install -q ultralytics==8.4.86

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Google Drive의 `MyDrive/cobot3_yolo/` 폴더에 다음 두 파일을 올려두세요.

- `pose_dataset.zip`
- `prepare_dataset.py`

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/cobot3_yolo')
ZIP_PATH = DRIVE_DIR / 'pose_dataset.zip'
PREPARE_SCRIPT = DRIVE_DIR / 'prepare_dataset.py'
DATASET_DIR = Path('/content/bolt_seg')
RUNS_DIR = DRIVE_DIR / 'runs'

assert ZIP_PATH.is_file(), f'파일 없음: {ZIP_PATH}'
assert PREPARE_SCRIPT.is_file(), f'파일 없음: {PREPARE_SCRIPT}'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('준비 완료:', DRIVE_DIR)

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, str(PREPARE_SCRIPT),
    '--source', str(ZIP_PATH),
    '--output', str(DATASET_DIR),
    '--overwrite',
], check=True)
print((DATASET_DIR / 'data.yaml').read_text())

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8n-seg.pt')
results = model.train(
    data=str(DATASET_DIR / 'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=8,
    device=device,
    workers=2,
    project=str(RUNS_DIR),
    name='bolt_seg',
    save_period=10,
    plots=True,
)
print('학습 결과:', results.save_dir)

In [ ]:
from shutil import copy2

best_path = Path(results.save_dir) / 'weights' / 'best.pt'
final_path = DRIVE_DIR / 'bolt_seg_best.pt'
copy2(best_path, final_path)
trained = YOLO(str(final_path))
metrics = trained.val(data=str(DATASET_DIR / 'data.yaml'), device=device)
print('classes:', trained.names)
print('saved:', final_path)

학습이 끝나면 Drive의 `MyDrive/cobot3_yolo/bolt_seg_best.pt`를 내려받아 `detection.py`의 `MODEL_PATH`에 연결하세요.